In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [2]:
mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

In [3]:
trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=train_transform
)
testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=test_transform
)

batch_size = 128
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
testloader  = DataLoader(testset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print("Train size:", len(trainset), "Test size:", len(testset))

Train size: 50000 Test size: 10000


In [4]:
images, labels = next(iter(trainloader))
print("Images:", images.shape, images.dtype)
print("Labels:", labels.shape, labels.dtype)
print("Label example:", labels[:10])

C:\Users\ben21\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Images: torch.Size([128, 3, 32, 32]) torch.float32
Labels: torch.Size([128]) torch.int64
Label example: tensor([6, 3, 2, 6, 3, 4, 5, 8, 9, 2])


In [5]:
class BasicCNN_BN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)
        
model = BasicCNN_BN().to(device)

In [6]:
images, labels = images.to(device), labels.to(device)
logits = model(images)
print("Logits shape:", logits.shape)


Logits shape: torch.Size([128, 10])


In [7]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

loss = criterion(logits, labels)
print("One batch loss:", loss.item())


One batch loss: 2.2858972549438477


In [8]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total

In [9]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total

In [ ]:
epochs = 15
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, trainloader)
    test_loss, test_acc = evaluate(model, testloader)

    print(f"Epoch {epoch:02d}/{epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
          f"Test Loss: {test_loss:.4f} Acc: {test_acc*100:.2f}%")

Epoch 01/15 | Train Loss: 1.5884 Acc: 41.38% | Test Loss: 1.2346 Acc: 54.96%
Epoch 02/15 | Train Loss: 1.3277 Acc: 51.38% | Test Loss: 1.1230 Acc: 59.67%
Epoch 03/15 | Train Loss: 1.2157 Acc: 56.18% | Test Loss: 1.0270 Acc: 63.31%
Epoch 04/15 | Train Loss: 1.1551 Acc: 58.49% | Test Loss: 0.9374 Acc: 67.31%
Epoch 05/15 | Train Loss: 1.1094 Acc: 60.35% | Test Loss: 0.8857 Acc: 68.80%
Epoch 06/15 | Train Loss: 1.0681 Acc: 61.64% | Test Loss: 0.8984 Acc: 68.89%
Epoch 07/15 | Train Loss: 1.0354 Acc: 63.21% | Test Loss: 0.8351 Acc: 70.35%
Epoch 08/15 | Train Loss: 1.0072 Acc: 64.37% | Test Loss: 0.8710 Acc: 68.65%
Epoch 09/15 | Train Loss: 0.9870 Acc: 65.12% | Test Loss: 0.8233 Acc: 71.36%
Epoch 10/15 | Train Loss: 0.9643 Acc: 65.93% | Test Loss: 0.8194 Acc: 71.18%
Epoch 11/15 | Train Loss: 0.9473 Acc: 66.59% | Test Loss: 0.7768 Acc: 72.41%
Epoch 12/15 | Train Loss: 0.9241 Acc: 67.31% | Test Loss: 0.7734 Acc: 72.99%
